In [1]:
#Retrieve the dataset with test summaries by human and T5
%store -r test_paired_summaries
%store -r t5_test_sumaries


In [2]:
import sys

!{sys.executable} -m pip install -q rouge-score bert-score scikit-learn
!{sys.executable} -m pip install bert-score
!{sys.executable} -m pip install torch --index-url https://download.pytorch.org/whl/cpu


Looking in indexes: https://download.pytorch.org/whl/cpu


In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import numpy as np

from bert_score import score as bertscore

import torch

from transformers import AutoModel


/Users/chadadelman/anaconda3/envs/chad_env/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [9]:
#rouge and cosine similarity methods (created with the help of ChatGPT)
def compute_rouge(text1, text2):
    """
    Compute ROUGE between two texts.
    Returns F1 scores for ROUGE-1, ROUGE-2, ROUGE-L.
    """
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=True
    )
    # Note: scorer.score(reference, prediction)
    scores = scorer.score(text2, text1)

    return {
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
    }


def compute_cosine_similarity(text1, text2):
    """
    Compute cosine similarity between two texts using TF-IDF vectors.
    This avoids loading any transformer model and is stable in most environments.
    """
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([text1, text2])  # shape (2, vocab_size)

    cos_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])  # (1,1)
    return float(cos_sim[0, 0])

def compute_bertscore(text1: str, text2: str):
    """
    Compute BERTScore (Precision, Recall, F1) between two texts.
    text1 = candidate
    text2 = reference
    """
    P, R, F1 = bertscore(
        cands=[text1],
        refs=[text2],
        lang="en"
    )
    return {
        "precision": float(P[0]),
        "recall": float(R[0]),
        "f1": float(F1[0]),
    }


In [12]:
import warnings
warnings.filterwarnings("ignore", message="Some weights of RobertaModel")

In [13]:
#For Testing
text_a = "How old are you going to be tomorrow?"
text_b = "What will your age be on the day that follows today"

print(compute_rouge(text_a, text_b))
print(compute_cosine_similarity(text_a, text_b))
print(compute_bertscore(text_a, text_b))

{'rouge1': 0.10526315789473685, 'rouge2': 0.0, 'rougeL': 0.10526315789473685}
0.05700655341923561


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'precision': 0.8961870670318604, 'recall': 0.8896476030349731, 'f1': 0.8929053544998169}
